# Model 3: SentenceTransformer End-to-End Fine-tuning

**Architecture:** all-MiniLM-L6-v2 (fine-tuned, 22M params) → mean pooling (384-dim) → regression head  
**Key difference from Model 1:** Encoder is NOT frozen — trained end-to-end on price data  

```
Model 1: all-MiniLM-L6-v2 (FROZEN) → pre-computed 384-dim → DNN head
Model 3: all-MiniLM-L6-v2 (FINE-TUNED) → batch-wise 384-dim → DNN head
```

**Hypothesis:** Frozen SentTrans was optimized for semantic similarity, not price prediction.  
Fine-tuning teaches the encoder to attend to brand names, materials, and category keywords.

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
```

In [1]:
from pricer.items import Item
from pricer.senttrans_e2e_model import SentTransE2ERunner
from pricer.evaluator import evaluate, plot_training_history

## 1. Load Data

In [2]:
train, val, test = Item.from_hub("SeanSunny/items_full")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

Train: 800,000 | Val: 10,000 | Test: 10,000


## 2. Setup Model

Loads all-MiniLM-L6-v2 with requires_grad=True. No pre-computation — encoder runs in each batch.

In [3]:
runner = SentTransE2ERunner(train, val[:1000])
runner.setup(batch_size=256)

Loading tokenizer from sentence-transformers/all-MiniLM-L6-v2...


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Loading SentTrans E2E model (encoder will be fine-tuned)...


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

SentTrans E2E: 22,812,801 params (encoder: 22,713,216, head: 99,585)
Using cuda


## 3. Train

Max 15 epochs, early stopping patience=3.  
Discriminative LR: encoder=5e-5 (preserve pretrained), head=1e-3.

In [4]:
history = runner.train(epochs=15, patience=3, warmup_steps=500)

Epoch 1/15:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch [1/15]
  Train Loss: 0.4806, Val Loss: 0.4466
  Val MAE: $62.59, LR: 0.00004717
  ** New best Val MAE: $62.59


Epoch 2/15:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch [2/15]
  Train Loss: 0.4081, Val Loss: 0.4158
  Val MAE: $57.35, LR: 0.00004380
  ** New best Val MAE: $57.35


Epoch 3/15:   0%|          | 0/3125 [00:00<?, ?it/s]

...

Epoch [13/15]
  Train Loss: 0.2675, Val Loss: 0.3458
  Val MAE: $47.01, LR: 0.00000674
  ** New best Val MAE: $47.01


Epoch 14/15:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch [14/15]
  Train Loss: 0.2626, Val Loss: 0.3453
  Val MAE: $47.70, LR: 0.00000337
  No improvement (1/3)


Epoch 15/15:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch [15/15]
  Train Loss: 0.2587, Val Loss: 0.3452
  Val MAE: $47.21, LR: 0.00000000
  No improvement (2/3)


## 4. Training History

In [5]:
plot_training_history(history, title="SentTrans E2E Fine-tuning")

## 5. Evaluate on 200 Test Samples

In [6]:
evaluate(runner.inference, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$67 $95 $12 $34 $23 $117 $56 $35 $4 $137 $85 $192 $3 $6 $12 $4 $43 $23 $22 $73 $34 $36 $4 $124 $30 $221 $66 $3 $80 $58 $48 $15 $66 $27 $11 $218 $73 $33 $101 $4 $17 $37 $3 $4 $46 $6 $9 $6 $76 $5 $7 $46 $133 $50 $24 $13 $16 $190 $39 $3 $126 $38 $26 $56 $256 $13 $7 $238 $6 $63 $16 $1 $55 $12 $24 $4 $46 $1 $0 $3 $16 $19 $0 $59 $10 $120 $83 $263 $18 $12 $6 $6 $3 $3 $1 $52 $8 $31 $23 $180 $12 $21 $3 $0 $12 $102 $2 $297 $6 $48 $23 $20 $5 $40 $3 $25 $24 $2 $81 $185 $7 $19 $6 $29 $43 $58 $0 $26 $64 $63 $33 $66 $3 $2 $160 $0 $119 $29 $27 $26 $12 $93 $19 $4 $9 $1 $9 $217 $44 $8 $1 $3 $11 $22 $98 $29 $43 $6 $34 $9 $64 $17 $5 $0 $306 $2 $178 $28 $12 $1 $24 $6 $205 $20 $37 $22 $1 $28 $65 $10 $162 $3 $58 $21 $4 $23 $84 $8 $23 $16 $6 $1 $13 $33 $8 $10 $41 $12 $22 $13 

## 6. Save Model Weights

In [7]:
runner.save("senttrans_e2e_model.pth")
print("Saved to senttrans_e2e_model.pth")

Saved to senttrans_e2e_model.pth


## 7. Sanity Check

In [8]:
sample = test[0]
pred = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  ${sample.price:.2f}")
print(f"Predict: ${pred:.2f}")
print(f"Error:   ${abs(pred - sample.price):.2f}")

Product: Old Blood Noise Excess V2 Distortion Chorus/Delay Pedal
Actual:  $219.00
Predict: $286.12
Error:   $67.12
